In [1]:
# CÉLULA 1 — Imports (atualizada)
from pyspark.sql.functions import (
    from_json, col, from_unixtime, to_timestamp,
    dayofweek, dayofmonth, hour, current_timestamp,
    when, date_format, max, sha2, concat_ws
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 3, Finished, Available, Finished, False)

In [2]:
# CÉLULA 2 — Criar tabela etl_state se não existir
spark.sql("""
    CREATE TABLE IF NOT EXISTS etl_state (
        pipeline_name STRING,
        last_watermark TIMESTAMP
    )
""")

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
# CÉLULA 3 — Ler o watermark atual
watermark_df = spark.sql("SELECT last_watermark FROM etl_state WHERE pipeline_name = 'silver_sensors'")

if watermark_df.count() > 0:
    last_watermark = watermark_df.collect()[0][0]
else:
    last_watermark = '1900-01-01'

print(f"Watermark atual: {last_watermark}")

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 5, Finished, Available, Finished, False)

Watermark atual: 2026-04-09 05:24:18


In [4]:
# CÉLULA 4 — Ler e fazer parse do Bronze
df = spark.read.table("bronze_raw")

schema = StructType([
    StructField("mac", StringType()),
    StructField("temperature", DoubleType()),
    StructField("humidity", DoubleType()),
    StructField("battery", DoubleType()),
    StructField("timestamp", StructType([
        StructField("$date", LongType())
    ]))
])

df_parsed = df.withColumn("parsed", from_json(col("payload.after"), schema))

df_silver = df_parsed.select(
    sha2(concat_ws("_", col("parsed.mac"), col("parsed.timestamp.`$date`").cast("string")), 256).alias("event_id"),
    col("parsed.mac").alias("mac"),
    col("parsed.temperature").alias("temperature"),
    col("parsed.humidity").alias("humidity"),
    when(
        (col("parsed.battery") >= 0) & (col("parsed.battery") <= 100),
        col("parsed.battery")
    ).otherwise(None).alias("battery"),
    col("parsed.timestamp.`$date`").alias("timestamp_ms"),
    to_timestamp(from_unixtime(col("parsed.timestamp.`$date`") / 1000)).alias("event_timestamp"),
    dayofweek(to_timestamp(from_unixtime(col("parsed.timestamp.`$date`") / 1000))).alias("day_of_week"),
    dayofmonth(to_timestamp(from_unixtime(col("parsed.timestamp.`$date`") / 1000))).alias("day_of_month"),
    hour(to_timestamp(from_unixtime(col("parsed.timestamp.`$date`") / 1000))).alias("hour"),
    current_timestamp().alias("ingestion_timestamp"),
    when(
        (col("parsed.temperature") >= -10) & (col("parsed.temperature") <= 50) &
        (col("parsed.humidity") >= 10) & (col("parsed.humidity") <= 90),
        True
    ).otherwise(False).alias("is_valid"),
    date_format(
        to_timestamp(from_unixtime(col("parsed.timestamp.`$date`") / 1000)),
        "EEEE"
    ).alias("day_name")
)

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 6, Finished, Available, Finished, False)

In [5]:
# CÉLULA 5 — Filtrar dados novos e remover duplicados (atualizada)
df_new = df_silver.filter(col("event_timestamp") > last_watermark)

# Anti-join: remover registos que já existem no Silver
try:
    existing_ids = spark.sql("SELECT event_id FROM silver_sensors")
    df_new = df_new.join(existing_ids, on="event_id", how="left_anti")
except:
    pass  # Tabela ainda não existe na primeira execução

new_count = df_new.count()
print(f"Registos novos encontrados: {new_count}")

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 7, Finished, Available, Finished, False)

Registos novos encontrados: 1792


In [6]:
if new_count > 0:
    df_new.write.format("delta").mode("append").saveAsTable("silver_sensors")
    
    new_watermark = df_new.select(max("event_timestamp")).collect()[0][0]
    
    if new_watermark is not None:
        new_watermark_str = new_watermark.strftime("%Y-%m-%d %H:%M:%S")
        
        if watermark_df.count() > 0:
            spark.sql(f"UPDATE etl_state SET last_watermark = '{new_watermark_str}' WHERE pipeline_name = 'silver_sensors'")
        else:
            spark.sql(f"INSERT INTO etl_state VALUES ('silver_sensors', '{new_watermark_str}')")
        
        print(f"Registos escritos no Silver: {new_count}")
        print(f"Watermark atualizado para: {new_watermark_str}")
    else:
        print("Watermark inválido, etl_state mantido.")
else:
    print("Sem registos novos. Silver e watermark mantidos.")

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 8, Finished, Available, Finished, False)

Watermark inválido, etl_state mantido.


In [7]:
spark.sql("SELECT MIN(battery), MAX(battery) FROM silver_sensors WHERE battery IS NOT NULL").show()

StatementMeta(, 94aa1561-5f4f-4d42-939c-e347eddba9f5, 9, Finished, Available, Finished, False)

+------------+------------+
|min(battery)|max(battery)|
+------------+------------+
|         0.0|       100.0|
+------------+------------+

